In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# BLIP-2 Fine-Tuning — Pics Can Lie

**Model:** `Salesforce/blip2-opt-2.7b`  
**Task:** Text generation — predict `real` or `out-of-context`  
**Dataset:** `D:/Pics Can Lie/merged_balanced/train.json`

## 1 — Config

In [ ]:
import json
import os
import time

import torch
from PIL import Image
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from transformers import AutoProcessor, Blip2ForConditionalGeneration

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID        = "Salesforce/blip2-opt-2.7b"
ANNOTATIONS     = r"D:\Pics Can Lie\dataset\data\NewsClipPings\merged_balanced\train.json"
METADATA        = r"D:\Pics Can Lie\dataset\data\NewsClipPings\metadata\train.json"
IMAGES_ROOT     = r"D:\Pics Can Lie\dataset\origin"
OUTPUT_DIR      = r"D:\Pics Can Lie\blip2_finetuned"
MAX_SAMPLES     = 5000
BATCH_SIZE      = 2
GRAD_ACCUM      = 8          # effective batch = 16
EPOCHS          = 2
LR_QFORMER      = 1e-5
LR_VISION       = 1e-6
VISION_UNFREEZE = 2          # last N layers of vision encoder to unfreeze
LOG_EVERY       = 50         # steps
MAX_TARGET_LEN  = 8          # 'real' or 'out-of-context' + EOS
MAX_INPUT_LEN   = 128

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Config loaded. Output dir:", OUTPUT_DIR)

## 2 — Dataset

In [ ]:
def resolve_image_path(meta_image_path: str, images_root: str) -> str:
    rel = meta_image_path.replace("visual_news/", "", 1)
    return os.path.join(images_root, rel)


class PicsCanLieDataset(Dataset):
    def __init__(self, annotations_path: str, metadata_path: str,
                 images_root: str, max_samples: int):
        with open(annotations_path, "r", encoding="utf-8") as f:
            annotations = json.load(f)["annotations"]
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)  # keyed by image_id (string)

        joined = []
        for ann in annotations:
            image_id = str(ann["image_id"])
            if image_id not in metadata:
                continue
            meta    = metadata[image_id]
            caption = meta.get("caption") or meta.get("title") or ""
            joined.append({
                "image_path": resolve_image_path(meta["image_path"], images_root),
                "caption":    caption,
                "label_text": "out-of-context" if ann["falsified"] else "real",
            })

        self.items = joined[:max_samples]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        try:
            image = Image.open(item["image_path"]).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224), color=(128, 128, 128))

        prompt = (
            f"Does this image match the caption: '{item['caption'][:100]}'? "
            "Answer:"
        )
        return image, prompt, item["label_text"]


def collate_fn(batch, processor):
    images, prompts, labels = zip(*batch)

    inputs = processor(
        images=list(images),
        text=list(prompts),
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LEN,
    )

    label_enc = processor.tokenizer(
        list(labels),
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_TARGET_LEN,
    )
    label_ids = label_enc["input_ids"].clone()
    # Replace padding token id with -100 so loss ignores padding
    label_ids[label_ids == processor.tokenizer.pad_token_id] = -100

    inputs["labels"] = label_ids
    return inputs


print("Dataset class defined.")

## 3 — Load Model & Processor

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

print("Loading processor …")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loading model …")
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
model.to(device)
print("Model loaded.")

Device: cuda
Loading processor …


d:\Pics Can Lie\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Youssef Elghandour\.cache\huggingface\hub\models--Salesforce--blip2-opt-2.7b. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the mo

Loading model …


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

## 4 — Freeze / Unfreeze Strategy

In [ ]:
# Freeze everything first
for p in model.parameters():
    p.requires_grad = False

# Unfreeze Q-Former + language_projection (LR_QFORMER)
for p in model.qformer.parameters():
    p.requires_grad = True
for p in model.language_projection.parameters():
    p.requires_grad = True

# Unfreeze last VISION_UNFREEZE layers of vision encoder (LR_VISION)
encoder_layers = model.vision_model.encoder.layers
for layer in encoder_layers[-VISION_UNFREEZE:]:
    for p in layer.parameters():
        p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## 5 — Optimizer & DataLoader

In [ ]:
qformer_params = [p for p in list(model.qformer.parameters()) +
                                list(model.language_projection.parameters())
                  if p.requires_grad]

vision_params = [p for layer in encoder_layers[-VISION_UNFREEZE:]
                   for p in layer.parameters()
                   if p.requires_grad]

optimizer = torch.optim.AdamW([
    {"params": qformer_params, "lr": LR_QFORMER},
    {"params": vision_params,  "lr": LR_VISION},
])

print("Loading dataset …")
dataset = PicsCanLieDataset(ANNOTATIONS, METADATA, IMAGES_ROOT, MAX_SAMPLES)
print(f"Samples: {len(dataset)}")

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=lambda b: collate_fn(b, processor),
    pin_memory=torch.cuda.is_available(),
)
print(f"Batches per epoch: {len(loader)}")

## 6 — Training Loop

In [ ]:
def vram_gb() -> float:
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 ** 3
    return 0.0


scaler = GradScaler(enabled=torch.cuda.is_available())

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    epoch_loss = 0.0
    t0 = time.time()

    for step, batch in enumerate(loader, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with autocast(enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss    = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        if step % GRAD_ACCUM == 0 or step == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        epoch_loss += loss.item() * GRAD_ACCUM  # un-scale for logging

        if step % LOG_EVERY == 0:
            avg_loss = epoch_loss / step
            elapsed  = time.time() - t0
            print(
                f"Epoch {epoch} | step {step}/{len(loader)} "
                f"| loss {avg_loss:.4f} "
                f"| VRAM {vram_gb():.2f} GB "
                f"| {elapsed:.0f}s elapsed"
            )

    avg_epoch_loss = epoch_loss / len(loader)
    print(f"\nEpoch {epoch} complete — avg loss {avg_epoch_loss:.4f}\n")

    # Save checkpoint
    ckpt_dir = os.path.join(OUTPUT_DIR, f"epoch_{epoch}")
    os.makedirs(ckpt_dir, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)
    print(f"Checkpoint saved → {ckpt_dir}\n")

print("Training complete.")